# BODAQS Spatial Context Explorer — Self-scoped

Prototype distance-domain preprocessing and visualization for one processed session. Derivation is in memory: suspension motion is calculated from the full-resolution, low-pass-filtered wheel-displacement samples and only then aggregated onto the spatial grid.

## 1. Configure library

Set `LIBRARIES_ROOT` and `LIBRARY_ID`, then run the notebook top-to-bottom.

In [1]:
from pathlib import Path
import copy
import sys

from IPython.display import display
import ipywidgets as W
import pandas as pd
import plotly.io as pio


def find_analysis_dir(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'bodaqs_analysis').is_dir():
            return candidate
        analysis = candidate / 'analysis'
        if (analysis / 'bodaqs_analysis').is_dir():
            return analysis
    raise RuntimeError('Could not find the BODAQS analysis package root.')


ANALYSIS_DIR = find_analysis_dir()
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

LIBRARIES_ROOT = Path.home() / 'OneDrive' / 'BODAQS-data'
LIBRARY_ID = 'default-library'

from bodaqs_analysis.library_api import LibraryAdapter

adapter = LibraryAdapter(LIBRARIES_ROOT)
libraries = {item['library_id']: item for item in adapter.list_libraries()}
if LIBRARY_ID not in libraries:
    available = ', '.join(sorted(libraries)) or 'none found'
    raise ValueError(f'Library {LIBRARY_ID!r} was not found. Available: {available}')
LIBRARY_ROOT = Path(libraries[LIBRARY_ID]['root'])
pio.renderers.default = 'notebook_connected'
print(f'Analysis package root: {ANALYSIS_DIR}')
print(f'Library root: {LIBRARY_ROOT}')

Analysis package root: C:\Users\benco\dev\BODAQS\analysis
Library root: C:\Users\benco\OneDrive\BODAQS-data\libraries\default-library


## 2. Select one processed session

The session must contain usable GPS/FIT position evidence. Front and rear activity additionally require full-resolution, filtered wheel-domain displacement signals.

In [2]:
from bodaqs_analysis.widgets.session_selector import make_session_selector

sel = make_session_selector(
    artifacts_dir=LIBRARY_ROOT,
    include_aggregations=False,
    select_first_by_default=True,
    autosave_default=False,
)
display(sel['ui'])

## 3. Set exploratory parameters

Every current parameter is exposed here. The 1 Hz and 99% GPS thresholds are advisory while `quality_action` remains `warn`. Edit this dictionary and rerun the derivation cell to compare settings.

In [15]:
from bodaqs_analysis.spatial_context import (
    DEFAULT_SPATIAL_CONTEXT_CONFIG,
    DEFAULT_SPATIAL_CONTEXT_TRACK_SCOPE_CONFIG,
)

SPATIAL_CONTEXT_CONFIG = copy.deepcopy(DEFAULT_SPATIAL_CONTEXT_CONFIG)
SPATIAL_CONTEXT_CONFIG.update({
    'enabled': True,
    'distance': {
        'source_priority': ['gps_geometry'],
        'grid_interval_m': 0.5,
        'distance_model': 'geodesic',
        'max_interpolation_gap_s': 5.0,
        'minimum_nominal_gps_rate_hz': 1.0,
        'minimum_gps_coverage_ratio': 0.99,
        'minimum_distance_support_fraction': 0.5,
        'maximum_implied_speed_mps': 50.0,
        'quality_action': 'warn',
        'geometry_denoising': {
            'enabled': True,
            'estimator': 'local_polynomial',
            'window_m': 20.0,
            'polynomial_order': 2,
            'fit_weighting': 'tricube',
            'robust_iterations': 2,
            'robust_tuning_constant': 4.685,
        },
    },
    'gradient': {
        'enabled': True,
        'altitude_source': 'gps',
        'estimator': 'local_linear_regression',
        'regression_window_m': 20.0,
        'smoothing_kernel': 'centred_exponential',
        'smoothing_distance_m': 15.0,
    },
    'twistiness': {
        'enabled': True,
        'estimator': 'local_polynomial',
        'geometry_window_m': 20.0,
        'polynomial_order': 2,
        'require_full_window': True,
        'minimum_source_position_observations': 3,
        'fit_weighting': 'tricube',
        'horizontal_accuracy_weighting': True,
        'horizontal_accuracy_floor_m': 0.5,
        'robust_iterations': 2,
        'robust_tuning_constant': 4.685,
        'smoothing_kernel': 'centred_exponential',
        'smoothing_distance_m': 7.5,
    },
    'suspension_activity': {
        'enabled': True,
        'use_preprocess_active_mask': True,
        'front_selector': {'end': 'front', 'quantity': 'disp', 'domain': 'wheel', 'unit': 'mm', 'processing_role': 'primary_analysis'},
        'rear_selector': {'end': 'rear', 'quantity': 'disp', 'domain': 'wheel', 'unit': 'mm', 'processing_role': 'primary_analysis'},
        'minimum_support_fraction': 0.25,
        'smoothing_kernel': 'centred_exponential',
        'smoothing_distance_m': 4.0,
        'combined_method': 'mean_both_required',
    },
})

# Applied only after the whole-session metrics have been derived, and only
# when a track is selected below. All matching parameters remain exposed.
TRACK_SCOPE_CONFIG = copy.deepcopy(DEFAULT_SPATIAL_CONTEXT_TRACK_SCOPE_CONFIG)
TRACK_SCOPE_CONFIG.update({
    'traversal_selection': 'last_forward_traversal',
    'matching': {
        'maximum_lateral_distance_m': 8.0,
        'maximum_match_gap_s': 5.0,
        'endpoint_tolerance_m': 15.0,
        'minimum_track_coverage_ratio': 0.85,
        'minimum_forward_fraction': 0.60,
        'projection_candidate_count': 8,
        'projection_station_separation_m': 5.0,
        'transition_distance_weight': 0.5,
        'heading_alignment_weight': 2.0,
    },
})
SPATIAL_CONTEXT_CONFIG, TRACK_SCOPE_CONFIG

{'enabled': True,
 'algorithm_version': 2,
 'distance': {'source_priority': ['gps_geometry'],
  'grid_interval_m': 0.5,
  'distance_model': 'geodesic',
  'max_interpolation_gap_s': 5.0,
  'minimum_nominal_gps_rate_hz': 1.0,
  'minimum_gps_coverage_ratio': 0.99,
  'minimum_distance_support_fraction': 0.5,
  'maximum_implied_speed_mps': 50.0,
  'quality_action': 'warn',
  'geometry_denoising': {'enabled': True,
   'estimator': 'local_polynomial',
   'window_m': 20.0,
   'polynomial_order': 2,
   'fit_weighting': 'tricube',
   'robust_iterations': 2,
   'robust_tuning_constant': 4.685}},
 'gradient': {'enabled': True,
  'altitude_source': 'gps',
  'estimator': 'local_linear_regression',
  'regression_window_m': 20.0,
  'smoothing_kernel': 'centred_exponential',
  'smoothing_distance_m': 15.0},
 'twistiness': {'enabled': True,
  'estimator': 'local_polynomial',
  'geometry_window_m': 20.0,
  'polynomial_order': 2,
  'require_full_window': True,
  'minimum_source_position_observations': 3,


## 4. Derive, inspect, and map a distance selection to time

Press **Derive spatial context** after changing the config. Optionally select a saved track to retain one matched forward traversal; `last_forward_traversal` is the default. Track selection is a post-derivation scope operation: every metric still comes from the selected session. The distance range is display-only and reports separate time intervals when missing evidence prevents a safe continuous mapping.

In [16]:
from bodaqs_analysis.artifacts import load_session_artifacts
from bodaqs_analysis.dashboards.spatial_context import (
    available_spatial_context_metrics,
    make_spatial_context_figure,
    spatial_selection_to_time_ranges,
)
from bodaqs_analysis.spatial_context import derive_spatial_context, scope_spatial_context_to_track

tracks = adapter.list_tracks()
tracks_by_id = {track['track_id']: track for track in tracks}
track_select = W.Dropdown(
    description='Track',
    options=[('None — full session', '')] + [
        (f"{track.get('display_name') or track['track_id']} (r{track.get('revision', '?')})", track['track_id'])
        for track in tracks
    ],
    layout=W.Layout(width='520px'),
)
derive_button = W.Button(description='Derive spatial context', button_style='primary')
render_button = W.Button(description='Render selected view')
metrics_select = W.SelectMultiple(description='Metrics', rows=5, layout=W.Layout(width='430px'))
show_local = W.Checkbox(value=True, description='Show local estimates', indent=False)
start_distance = W.FloatText(description='Start [m]')
end_distance = W.FloatText(description='End [m]')
max_time_gap = W.BoundedFloatText(value=5.0, min=0.01, max=300.0, step=0.5, description='Time gap [s]')
status_out = W.Output(layout=W.Layout(width='100%'))
plot_out = W.Output(layout=W.Layout(width='100%'))
SESSION_SPATIAL_RESULT = None
SPATIAL_RESULT = None


def derive_selected_session(_=None):
    global SESSION_SPATIAL_RESULT, SPATIAL_RESULT
    selected = list(sel['get_selected']())
    if len(selected) != 1:
        raise ValueError(f'Select exactly one physical session; selected {len(selected)}.')
    ref = selected[0]
    session = load_session_artifacts(sel['store'], run_id=ref['run_id'], session_id=ref['session_id'])
    secondary_meta = session.pop('secondary_stream_meta', {})
    session.setdefault('meta', {}).setdefault('secondary_streams', {}).update(secondary_meta)
    SESSION_SPATIAL_RESULT = derive_spatial_context(session, SPATIAL_CONTEXT_CONFIG)
    selected_track = tracks_by_id.get(track_select.value)
    SPATIAL_RESULT = (
        scope_spatial_context_to_track(SESSION_SPATIAL_RESULT, session, selected_track, TRACK_SCOPE_CONFIG)
        if selected_track is not None
        else SESSION_SPATIAL_RESULT
    )
    stream = SPATIAL_RESULT.stream_df
    available = available_spatial_context_metrics(stream) if not stream.empty else []
    metrics_select.options = available
    metrics_select.value = tuple(available)
    if not stream.empty:
        start_distance.value = float(stream['distance_m'].min())
        end_distance.value = float(stream['distance_m'].max())
    with status_out:
        status_out.clear_output()
        print(f"Run: {ref['run_id']} | Session: {ref['session_id']}")
        print(f"Status: {SPATIAL_RESULT.stream_meta['status']} | Spatial samples: {len(stream):,}")
        selected_source = SPATIAL_RESULT.stream_meta.get('distance_source', {}).get('selected')
        print('Distance source:', selected_source or 'unavailable')
        track_scope = SPATIAL_RESULT.stream_meta.get('track_scope')
        if track_scope:
            selected_traversal = track_scope.get('selected_traversal') or {}
            print(
                f"Track scope: {track_scope.get('track_ref')} | {track_scope.get('status')} | "
                f"policy: {track_scope.get('traversal_selection')} | "
                f"forward traversals: {track_scope.get('matching', {}).get('forward_traversal_count', 0)}"
            )
            if selected_traversal:
                print(
                    f"Selected traversal time: {selected_traversal.get('start_time_s'):.3f}–"
                    f"{selected_traversal.get('end_time_s'):.3f} s | "
                    f"track coverage: {selected_traversal.get('coverage_ratio'):.1%}"
                )
        warnings = SPATIAL_RESULT.stream_meta.get('warnings', [])
        print('Warnings: ' + (', '.join(warnings) if warnings else 'none'))
        if 'twistiness_source_observation_count' in stream:
            twistiness_valid = int(stream.get('twistiness_rad_per_m', pd.Series(dtype=float)).notna().sum())
            twistiness_provenance = SPATIAL_RESULT.stream_meta.get('metric_provenance', {}).get('twistiness', {})
            print(
                f"Twistiness valid rows: {twistiness_valid:,}/{len(stream):,} | "
                f"full window: {twistiness_provenance.get('effective_geometry_window_samples', 'n/a')} rows | "
                f"minimum source positions: {twistiness_provenance.get('minimum_source_position_observations', 'n/a')}"
            )
    render_spatial_view()


def render_spatial_view(_=None):
    if SPATIAL_RESULT is None or SPATIAL_RESULT.stream_df.empty:
        return
    metrics = list(metrics_select.value)
    if not metrics:
        raise ValueError('Select at least one available metric.')
    stream = SPATIAL_RESULT.stream_df
    distance_range = (start_distance.value, end_distance.value)
    ranges = spatial_selection_to_time_ranges(stream, *distance_range, max_time_gap_s=max_time_gap.value)
    with plot_out:
        plot_out.clear_output()
        make_spatial_context_figure(
            stream,
            metrics=metrics,
            show_local=show_local.value,
            selected_distance_range_m=distance_range,
        ).show()
        print('Representative time range(s):')
        display(pd.DataFrame(ranges))


derive_button.on_click(derive_selected_session)
render_button.on_click(render_spatial_view)
display(W.VBox([
    W.HBox([derive_button, render_button]),
    track_select,
    W.HBox([metrics_select, W.VBox([show_local, start_distance, end_distance, max_time_gap])]),
    status_out,
    plot_out,
]))

## Interpretation notes

- Gradient is dimensionless rise/run; multiply by 100 for percent grade.
- Twistiness is absolute horizontal curvature in radians per metre.
- Suspension activity is absolute wheel travel in metres per metre of valid active ground distance.
- Rear activity is absent unless a rear wheel-domain displacement signal resolves; shock motion is never substituted.
- Selecting a track does not source gradient, twistiness, or suspension activity from the track. The notebook derives the whole-session stream first, then selects and rebases the chosen session traversal. `session_distance_m`, `track_station_m`, and the traversal match diagnostics remain available in the scoped result.
- Track matching retains alternative projections at crossings and close parallel sections, then uses movement continuity and heading agreement to choose a coherent directed station sequence. The default scope is the last qualifying forward traversal.
- GPS-geometry distance and Workbench track stationing use the same exposed 20 m local-quadratic, tricube-weighted robust denoising policy. The notebook currently duplicates this policy pending integration into the main preprocessing pipeline.
- Twistiness fits local polynomials to independent source-GPS positions and evaluates curvature from their analytic derivatives. Interpolated spatial rows are not fitting evidence.
- Dotted local traces precede distance-domain smoothing. Edit the 20 m geometry window and 15 m output smoothing distance independently when testing sensitivity. A 15-30 m geometry-window sweep is a useful calibration range; 20 m is the recommended initial value for metre-class GPS.
- Tricube distance weighting, receiver horizontal-accuracy weighting, its accuracy floor, and robust-fit controls are exposed above. Accuracy weighting falls back safely when the selected source has no accuracy field.
- Twistiness is null unless a complete centred geometry window is available. With the current 0.5 m grid and 20 m window, runs shorter than 41 spatial rows are excluded and 10 m is trimmed from each discontinuity edge.
- A twistiness window must contain at least three distinct supported source-GPS positions. The wider recommended window normally supplies substantially more; the threshold remains a hard availability rule rather than a smoothing control.